# Kulima OS Prospectus Fix and Completion

This notebook inspects and repairs the Kulima OS prospectus generation pipeline. It covers backend endpoint behavior, report generation logic, structured output, branding, PDF/HTML generation, and frontend integration.

## 1. Import Libraries and Project Files

Load Python libraries for file inspection, JSON handling, and report generation validation. Read the backend `/generate-prospectus` implementation and `prospectus_generator.py` logic.

In [ ]:
import json
from pathlib import Path
from pprint import pprint

repo_root = Path('c:/Users/HP/Desktop/kulima-os-hackathon')
backend_prospectus = repo_root / 'backend' / 'api' / 'prospectus.py'
core_prospectus = repo_root / 'core' / 'prospectus' / 'prospectus_generator.py'

print('Backend prospectus path:', backend_prospectus)
print('Core prospectus generator path:', core_prospectus)
print('Files exist:', backend_prospectus.exists(), core_prospectus.exists())

## 2. Inspect `/generate-prospectus` Endpoint

Read the backend endpoint implementation and identify any broken return values or coordination checks.

In [ ]:
with open(backend_prospectus, 'r', encoding='utf-8') as f:
    backend_code = f.read()

print('\n'.join(backend_code.splitlines()[:220]))

## 3. Debug and Fix `prospectus_generator.py`

Inspect the generator logic, locate any missing imports or broken report formatting, and confirm the report is only generated when coordination data exists.

In [ ]:
with open(core_prospectus, 'r', encoding='utf-8') as f:
    generator_code = f.read()

print('\n'.join(generator_code.splitlines()[:140]))
print('generate_pdf definition present:', 'def generate_pdf' in generator_code)
print('defaultdict imported:', 'from collections import defaultdict' in generator_code)

## 4. Define Structured Report Schema

The prospectus should include:
- Title and branding
- Executive summary
- Activity breakdown
- Coordination analysis
- Infrastructure demand signals
- Investment opportunity
- Risk and confidence level
- Conclusion

We will verify that the generator constructs these sections in the prospectus dictionary.

In [ ]:
sections = [
    'executive_summary',
    'coordination_patterns',
    'energy_signals',
    'load_estimation',
    'settlement_and_infrastructure_analysis',
    'critical_load_protection',
    'sustainability_impact',
    'risk_and_governance',
    'flow_insights',
    'decision_recommendations',
    'long_term_coordination_insights',
    'regional_flow_analysis',
    'infrastructure_roadmap',
    'scenario_projections',
    'policy_maker_section',
    'investor_section',
    'infrastructure_planner_section',
    'deployment_readiness',
    'production_readiness',
    'infrastructure_planning_guidance',
    'social_reserve_policy',
    'ethics_compliance',
    'methodology'
]
for sec in sections:
    print(sec, sec in generator_code)

## 5. Implement Kulima-Branded HTML/PDF Output

The generator already includes PDF output. We need to ensure PDF styling and branding are present and that the frontend can download the generated PDF.

In [ ]:
with open(repo_root / 'frontend' / 'app' / 'page.jsx', 'r', encoding='utf-8') as f:
    page_code = f.read()

print('Generate button present:', 'Generate Investment Report' in page_code or 'Create Investment Report' in page_code)
print('download PDF link present:', 'Download PDF Report' in page_code)
print('generate-prospectus endpoint present:', '/generate-prospectus' in page_code)

## 6. Simulate Full Report Flow with Sample Signals

Validate the report generator with a representative confidence result and ensure the prospectus dictionary contains all major sections without empty fields.

In [ ]:
try:
    from core.prospectus.prospectus_generator import ProspectusGenerator
    from policy import compute_planning_reserve

    sample_pattern = {
        'activity_type': 'irrigation',
        'zone': 'MZUZU',
        'time_window': 'morning',
        'service_priority': 'productive',
        'pattern_persistence': 0.85,
        'pattern_stability': 0.8,
        'demand_rhythm': {'frequency': '6 of 7 cycles', 'stability_class': 'stable'},
        'stability_score': 0.8,
        'validation_strength': 'strong',
        'validation_details': 'human and telemetry aligned',
        'integrity_score': 0.82,
        'confidence_class': 'high',
        'coordination_confidence': 0.82,
        'bankability_note': 'Bankable under pilot assumptions',
        'rejected_signals': 0,
        'signal_count': 12,
        'validated_signals': 10,
        'unique_days': 5,
        'unique_senders': 8,
        'trust': {'action_allowed': True},
        'explanation': {
            'why_accepted': 'Pattern meets persistence and validation thresholds.',
            'why_rejected': 'No upstream pattern rejection.',
            'reserve_explanation': '25% planning reserve applied for critical communal loads.',
            'action_allowed_explanation': 'Action allowed because trust score exceeds threshold.',
            'human_readable': 'High-confidence coordination pattern for irrigation in Mzuzu morning window.'
        },
    }

    planning_reserve = compute_planning_reserve(12)
    generator = ProspectusGenerator()
    prospectus = generator.generate_prospectus(
        [sample_pattern],
        lundai_analysis={'flow_graph': {'nodes': [], 'edges': []}},
        metadata={'region': 'MZUZU', 'period': '7-cycle window', 'is_sample': False},
        planning_reserve=planning_reserve
    )
    print('Generated prospectus keys count:', len(prospectus))
    print('Top-level keys:', list(prospectus.keys())[:12])
    print('Executive summary exists:', 'executive_summary' in prospectus)
    print('Coordination patterns count:', len(prospectus.get('coordination_patterns', [])))
    pprint({k: prospectus[k] for k in ['prospectus_metadata', 'document_scope', 'critical_load_protection']})
except Exception as e:
    print('Error during simulation:', type(e).__name__, e)


## 7. Add Edge Case Handling and Confidence Level Checks

Ensure the system returns a clear message when no signals or no stable coordination patterns exist.

The backend should return a structured error response when `generate-prospectus` cannot produce a bankable report.